# Cinema Ticket Demand Forecasting: Decision-Support Solution
**ALGIONS AI LABS - AI & Machine Learning Assessment**  
**Domain:** Cinema Hall Ticket Booking  
**Target:** `tickets_sold` & `occupancy_rate` (Show-Level)  
**Forecast Horizon:** T-24h (24 hours prior to scheduled showtime)  
**Goal:** Decision support for screen allocation and operational staff scheduling.

## 1. Problem Formulation & Setup
- **Prediction Unit:** An individual screening event $(Screen, Showtime, Movie, Date)$.
- **Horizon:** T-24h. No features derived from post-show bookings or same-day walk-ins are used.
- **Physical Constraint:** $0 \le \text{tickets\_sold} \le \text{screen\_capacity}$. Predictions must never exceed physical capacity.
- **Cold-Start Condition:** Movies in their first 3 days of release have negligible historical show observations.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds and styling
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

from src.config import DATA_PATH, MODEL_PATH, METADATA_PATH
from src.features import RAW_FEATURE_COLS, TARGET_COL, chronological_train_val_test_split, add_engineered_features
from src.models import CapacityAdjustedBaseline, CapacityBoundedRegressor, compute_forecasting_metrics

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with {df.shape[0]} shows and {df.shape[1]} features.")
df.head(3)

## 2. Exploratory Data Analysis & Leakage Diagnostics

In [ ]:
# Quality checks: nulls, data types, physical capacity bounds
print("Missing values per column:\n", df.isnull().sum())
capacity_violations = (df['tickets_sold'] > df['screen_capacity']).sum()
print(f"Physical capacity violations in raw data: {capacity_violations}")

# Summary statistics
df[['tickets_sold', 'screen_capacity', 'occupancy_rate', 'ticket_price', 'days_since_release']].describe()

In [ ]:
# Occupancy distribution by genre and day type
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x='genre', y='occupancy_rate', hue='is_weekend', palette='Set2')
plt.title("Show Occupancy Rate by Genre (Weekday vs. Weekend)", fontweight='bold')
plt.xticks(rotation=20)
plt.ylabel("Occupancy Rate")
plt.tight_layout()
plt.show()

## 3. Chronological Train / Validation / Test Splitting
To prevent temporal lookahead leakage, we split strictly by time:
- **Train Set:** First 70% of chronological observations (5,103 shows)
- **Validation Set:** Next 15% (1,093 shows)
- **Test Set:** Final 15% (1,094 shows)

In [ ]:
train_df, val_df, test_df = chronological_train_val_test_split(df, 0.70, 0.15)

print(f"Train range: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df)} shows)")
print(f"Val range:   {val_df['date'].min()} to {val_df['date'].max()} ({len(val_df)} shows)")
print(f"Test range:  {test_df['date'].min()} to {test_df['date'].max()} ({len(test_df)} shows)")

X_train, y_train = train_df[RAW_FEATURE_COLS], train_df[TARGET_COL]
X_val, y_val = val_df[RAW_FEATURE_COLS], val_df[TARGET_COL]
X_test, y_test = test_df[RAW_FEATURE_COLS], test_df[TARGET_COL]

test_caps = test_df['screen_capacity'].values

## 4. Model Training & Comparison
We train and benchmark:
1. **Baseline:** `CapacityAdjustedBaseline` (Historical group occupancy capped by hall capacity)
2. **Candidate 1:** `RidgeRegression` (Capacity-bounded regularized linear model)
3. **Candidate 2:** `GradientBoosting` (HistGradientBoostingRegressor with non-linear feature interactions and physical capacity truncation)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

# Baseline
base_model = CapacityAdjustedBaseline()
base_model.fit(X_train, y_train)
base_preds = base_model.predict(X_test)
base_metrics = compute_forecasting_metrics(y_test.values, base_preds, test_caps)

# Model 1: Ridge
ridge_model = CapacityBoundedRegressor(base_estimator=Ridge(alpha=10.0, random_state=42))
ridge_model.fit(X_train, y_train)
ridge_preds = ridge_model.predict(X_test)
ridge_metrics = compute_forecasting_metrics(y_test.values, ridge_preds, test_caps)

# Model 2: Gradient Boosting
gb_model = CapacityBoundedRegressor(
    base_estimator=HistGradientBoostingRegressor(
        random_state=42, max_iter=160, learning_rate=0.07, max_leaf_nodes=31, min_samples_leaf=20
    )
)
gb_model.fit(X_train, y_train)
gb_preds = gb_model.predict(X_test)
gb_metrics = compute_forecasting_metrics(y_test.values, gb_preds, test_caps)

comp_df = pd.DataFrame([base_metrics, ridge_metrics, gb_metrics], index=["Baseline", "Ridge", "GradientBoosting"])
comp_df

## 5. Held-Out Error Analysis & Cold-Start Slicing
Examining performance across operational sub-groups: cold-start theatrical openings vs. mature runs.

In [ ]:
test_results = test_df.copy()
test_results['predicted_tickets'] = gb_preds
test_results['abs_error'] = np.abs(test_results['tickets_sold'] - test_results['predicted_tickets'])

slice_cold = test_results.groupby('is_cold_start').agg(
    shows=('show_id', 'count'),
    mae=('abs_error', 'mean'),
    avg_actual=('tickets_sold', 'mean'),
    avg_pred=('predicted_tickets', 'mean')
).reset_index()
slice_cold['category'] = slice_cold['is_cold_start'].map({1: 'Cold-Start (Days 0-3)', 0: 'Mature (>3 Days)'})
slice_cold

## 6. Real Inference Demonstration & Decision-Support Output

In [ ]:
from src.explain import explain_single_prediction

sample_show = {
    "movie_title": "Aegis Protocol",
    "genre": "Action_SciFi",
    "critic_rating": 8.7,
    "star_power": 9.5,
    "runtime_minutes": 140,
    "days_since_release": 1,
    "is_cold_start": 1,
    "screen_id": "Screen_1_IMAX",
    "screen_tier": "IMAX",
    "screen_capacity": 420,
    "ticket_price": 22.5,
    "time_slot": "Night_Prime",
    "show_hour": 20,
    "day_of_week": 5,
    "is_weekend": 1,
    "is_holiday": 0,
    "competing_releases_count": 4,
    "prebooking_pace_t24": 210,
    "historical_movie_avg_occupancy": 0.65,
}

sample_row = pd.DataFrame([sample_show])[RAW_FEATURE_COLS]
preds, lower, upper = gb_model.predict_with_intervals(sample_row, confidence=0.80)

decision_output = explain_single_prediction(sample_show, preds[0], lower[0], upper[0])
print(json.dumps(decision_output, indent=2))

## 7. Operational Recommendations & Governance
- **Human-in-the-Loop Oversight:** The model is an operational decision-support tool. Cinema managers have final authority to adjust screen sizes or staffing based on live conditions (such as severe weather, power issues, or local events).
- **Cold-Start Protocol:** For movies on Days 0–3, confidence intervals are expanded, and an on-call staffing reserve is scheduled to absorb unexpected demand surges.